# M5.0b Primer: Circuits, Kirchhoff's Laws and RC Circuits

Book pointer: chapter 10, sections 10.1 (electric circuits), 10.2 (circuits with capacitors), 10.3 (circuits as models).

Second of four primers for module 5. Primer 00a described current inside a conductor as a field: a current density $\mathbf{J} = \sigma\mathbf{E}$ that obeys $\nabla\cdot\mathbf{J} = 0$ in steady state. This primer throws most of that detail away on purpose. A circuit keeps only a handful of numbers, the potentials of the junctions and the currents in the wires, and two laws that tie them together. The result is a linear system of exactly the kind solved in module 2, and it is what notebook 02 builds for networks of any size. The last section adds capacitors, which bring time into the picture: the RC circuit is the first circuit whose state changes, and it is the starting point of notebook 03.

## 1. Warm-up

Try on paper first.

1. A 9 V battery's terminal voltage drops to 8 V when it drives 0.5 A. What is its internal resistance?
2. State Kirchhoff's two laws, and name the field equation each one comes from.
3. Give the combined resistance of $R_1$ and $R_2$ in series and in parallel. Which combination is always smaller than the smaller of the two?
4. A circuit has $N$ nodes. How many unknown potentials does nodal analysis solve for, and why not $N$?
5. Define the time constant of an RC circuit. What fraction of its final voltage does a charging capacitor reach after one time constant?
6. A capacitor is charged from a battery through a resistor. What fraction of the energy the battery delivers ends up stored in the capacitor? Does the answer depend on $R$?
7. When is a Wheatstone bridge balanced, and why does no current flow through the middle branch then?

## 2. The ideas

**A circuit is a model.** A real circuit is a set of conductors in space, and the full description of what happens in it is the field $\mathbf{E}(\mathbf{r})$ and the current density $\mathbf{J}(\mathbf{r})$ everywhere. That description is almost never needed. Wires have such a high conductivity that the potential is nearly the same everywhere along one, so a whole wire and everything soldered to it can be summarised by one number, its potential. Each component between two such junctions (a resistor, a battery, a capacitor) can be summarised by one current and a rule that relates that current to the potential difference across it. A circuit diagram is this reduced description: nodes carry potentials, branches carry currents, and the geometry has disappeared. Section 3.4 shows the reduction working in the other direction too, by modelling a continuous conducting sheet as a grid of resistors.

**Why two laws are enough.** The reduction keeps exactly two facts from the field theory. The first is charge conservation in steady state, $\nabla\cdot\mathbf{J} = 0$ (primer 00a, section on continuity). Integrate it over a small volume that encloses a junction and the divergence theorem turns it into: the currents flowing out of a node add up to zero. That is Kirchhoff's current law, also called the node law. The second fact is that the electrostatic field is conservative, $\nabla\times\mathbf{E} = 0$, so its line integral around any closed loop is zero (primer M1.0d). Walk around any closed loop of the circuit and add up the potential changes across the components: they cancel. That is Kirchhoff's voltage law, the loop law. Both laws are statements about the fields, written in the reduced variables.

**Emf: what pushes the current.** A loop of resistors alone carries no current, because a conservative field cannot keep charge circulating against friction: going once round a closed loop, the field gives back exactly the energy it took, and the resistance takes energy out at every step. Something non-electrostatic has to do work on the charges. In a battery it is chemistry: reactions at the electrodes move ions against the electric field. In a generator it is the magnetic force on charges in moving wires (chapter 14). The work per unit charge done by this non-electrostatic push, once round the loop, is the electromotive force, or emf, written $\mathcal{E}$ and measured in volts. Despite the name it is not a force. Inside an ideal source the emf exactly balances the electrostatic field and the source holds its terminals at a potential difference $\mathcal{E}$.

**Real sources have internal resistance.** The charge pushed through a battery also has to move through its electrolyte and electrodes, which have resistance. A real battery is therefore modelled as an ideal emf $\mathcal{E}$ in series with a small internal resistance $r$. When it delivers a current $I$, the voltage at its terminals is $\mathcal{E} - Ir$, which is why a car's headlights dim when the starter motor draws a large current. The model also explains why a source delivers the most power to a load whose resistance equals its own internal resistance (section 3.1).

**A circuit is a linear system.** Each resistor gives a linear relation between a current and two potentials (Ohm's law), and each node gives a linear relation between currents (the node law). Substituting one into the other gives one linear equation per node in the unknown node potentials: $\mathbf{G}\mathbf{V} = \mathbf{I}$, where $\mathbf{G}$ is a conductance matrix and $\mathbf{I}$ holds the currents injected at the nodes by sources. This is the same structure as the discrete Poisson equation of M2 primer 00b, and for a good reason: the five-point stencil is the node law for a grid of equal resistors, with potential playing the role of voltage. Writing the matrix from the circuit is mechanical, which is what makes nodal analysis the method computers use. Section 3.3 does it by hand for three nodes and then hands it to `emlib.circuits.mna`.

**Capacitors bring in time.** A capacitor is two conductors separated by an insulator. No charge crosses the gap, but charge piles up on one plate and drains from the other, and the potential difference across it is $Q/C$ (M2 primer 00c). In a circuit, the current into a capacitor is the rate at which its charge changes, $I = C\,dV/dt$. Put a capacitor in series with a resistor and a battery, and the current is large at first, when the capacitor is empty and the whole battery voltage sits across the resistor, then falls as the capacitor fills. The capacitor voltage approaches the battery voltage exponentially, with a time constant $\tau = RC$. Every circuit that contains capacitors (or, in primer 00c, inductors) has such time scales, and the state of the circuit is described by differential equations instead of algebraic ones.

**Where this is used.** Notebook 02 assembles and solves the nodal equations for networks of any size, checks them against hand-solved circuits such as the Wheatstone bridge and the ladder network, and uses a grid of resistors as a model of a continuous sheet. Notebook 03 integrates the RC equations numerically, compares them with the exponential solutions derived here, and then moves to inductors and alternating current, which primer 00c prepares. Everything below is the material those notebooks assume.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci
from emlib.integrators import rk4_ode

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

### 3.1 Emf, batteries and the simplest circuit

Take a battery with emf $\mathcal{E}$ and internal resistance $r$, and connect a resistor $R$ (the load) across its terminals. There is one loop. Going round it in the direction of the current, the potential rises by $\mathcal{E}$ across the ideal source and falls by $Ir$ across the internal resistance and by $IR$ across the load. The loop law says the changes cancel:
$$\mathcal{E} - Ir - IR = 0 \quad\Longrightarrow\quad I = \frac{\mathcal{E}}{R + r}.$$
The terminal voltage, the potential difference a voltmeter across the battery would read, is $V = \mathcal{E} - Ir = \mathcal{E}\,R/(R + r)$. With no load ($R \to \infty$) it equals the emf. With a short circuit ($R = 0$) it drops to zero and the current is limited only by $r$, which is why shorting a battery heats it.

Where does the energy go? The source does work $\mathcal{E}$ on each coulomb, so it delivers power $\mathcal{E}I$. The load dissipates $I^2R$ and the battery's own resistance $I^2r$, and $\mathcal{E}I = I^2R + I^2r$ by the loop equation. Energy conservation and the loop law are the same statement.

**Worked example 1 (a 9 V battery).** $\mathcal{E} = 9\,$V, $r = 1.5\,\Omega$, $R = 10\,\Omega$. The current is $I = 9/11.5 = 0.7826\,$A. The terminal voltage is $9 - 0.7826\cdot1.5 = 7.826\,$V, so a voltmeter reads more than a volt less than the label. The load receives $I^2R = 6.125\,$W and the battery warms itself with $I^2r = 0.919\,$W. Warm-up question 1 runs the other way: a drop of $1\,$V at $0.5\,$A means $r = 1/0.5 = 2\,\Omega$.

**Worked example 2 (maximum power transfer).** For a fixed source, which load receives the most power? $P(R) = I^2R = \mathcal{E}^2R/(R + r)^2$. Differentiate:
$$\frac{dP}{dR} = \mathcal{E}^2\,\frac{(R + r)^2 - 2R(R + r)}{(R + r)^4} = \mathcal{E}^2\,\frac{r - R}{(R + r)^3},$$
which vanishes at $R = r$. The maximum power is $P_{\max} = \mathcal{E}^2/4r$, here $81/6 = 13.5\,$W. At that point the efficiency $R/(R + r)$ is only 50 percent: half the power heats the battery. That is why matching is used where the signal is weak and power is scarce (antennas, audio amplifiers), but never in the power grid, where the load resistance is made much larger than the source's so that little is lost in the source.

A note on the model. The ideal emf plus internal resistance is a linear model of a source. It works over the range of currents where the battery's chemistry keeps $\mathcal{E}$ and $r$ roughly constant, and it is the template for every source in this module. The code reproduces worked example 1, solves the same circuit with the general solver used in section 3.3, and plots $P(R)$.

In [ ]:
emf, r, R = 9.0, 1.5, 10.0
I = emf / (R + r)
print(f"I = {I:.4f} A   terminal voltage = {emf - I * r:.3f} V   power in the load = {I**2 * R:.3f} W   in the battery = {I**2 * r:.3f} W")

# The same circuit for the general solver: node 1 is inside the battery, node 2 its + terminal, node 0 ground.
V, _ = ci.mna(3, resistors=[(1, 2, r), (2, 0, R)], voltage_sources=[(1, 0, emf)])
print(f"mna: terminal potential = {V[2]:.3f} V")

Rs = np.linspace(0.01, 12, 4000)
P = emf**2 * Rs / (Rs + r) ** 2
k = np.argmax(P)
print(f"P(R) is largest at R = {Rs[k]:.3f} ohm (r = {r}),  P_max = {P[k]:.3f} W,  emf^2 / 4r = {emf**2 / (4 * r):.3f} W")

fig, ax = plt.subplots()
ax.plot(Rs, P, label="power in the load [W]")
ax.plot(Rs, 10 * Rs / (Rs + r), "--", label="efficiency R/(R+r), x10")
ax.axvline(r, color="grey", ls=":"); ax.set_xlabel("load resistance R [ohm]"); ax.legend(); ax.set_title("maximum power at R = r, efficiency 50 percent")
plt.show()

### 3.2 Kirchhoff's laws

**The node law.** Draw a small closed surface around a junction where several wires meet. In steady state no charge accumulates anywhere, so $\partial\rho/\partial t = 0$ and the continuity equation of primer 00a reduces to $\nabla\cdot\mathbf{J} = 0$. By the divergence theorem the net current out of the surface is zero. The surface cuts each wire once, and the current through it is that wire's current, so
$$\sum_{\text{branches at a node}} I_{\text{out}} = 0.$$
Even with capacitors present this holds at every node: a capacitor's plates gain and lose charge, but each node is a piece of wire, and the lumped model assumes wires store no charge. The charge that builds up sits on the plates, inside the component, where the model accounts for it through $I = C\,dV/dt$.

**The loop law.** In electrostatics $\nabla\times\mathbf{E} = 0$, so $\oint\mathbf{E}\cdot d\mathbf{l} = 0$ around any closed path, and the potential $V$ is single valued. Follow a closed loop of the circuit, from node to node through components, and add up the potential differences: you come back to the starting potential, so
$$\sum_{\text{components around a loop}} \Delta V = 0.$$
Inside a battery the electrostatic field points against the emf's push, so walking through a source from its $-$ to its $+$ terminal the potential rises by $\mathcal{E}$ (minus $Ir$ if it has internal resistance). Walking through a resistor in the direction of its current the potential falls by $IR$, because current flows downhill in potential.

**Sign conventions.** Choose a direction for each branch current before you know it. If the answer comes out negative, the current flows the other way; nothing else needs to be redone. Be consistent: in the loop law a resistor contributes $-IR$ when you walk along your chosen current direction and $+IR$ when you walk against it. Most sign errors in circuit analysis are a mixed convention in one loop.

**Series and parallel, from the two laws.** Two resistors in series carry the same current (node law at the junction between them, which has only two branches), and their voltages add (loop law), so $V = IR_1 + IR_2$ and $R_{\text{series}} = R_1 + R_2$. Two resistors in parallel share the same voltage (loop law around the small loop they form) and their currents add (node law), so $I = V/R_1 + V/R_2$ and
$$\frac{1}{R_{\text{par}}} = \frac{1}{R_1} + \frac{1}{R_2}, \qquad R_{\text{par}} = \frac{R_1R_2}{R_1 + R_2}.$$
Conductances $G = 1/R$ add in parallel the way resistances add in series. A parallel combination is always smaller than the smaller of the two resistors, because the second one only opens an extra path.

**Dividers.** Two resistors in series across a voltage $V_s$ split it in proportion: the voltage across $R_2$ is $V_s R_2/(R_1 + R_2)$, the voltage divider. A current $I_s$ fed into two resistors in parallel splits in inverse proportion: the current through $R_1$ is $I_s R_2/(R_1 + R_2)$, the current divider. Both appear constantly, including inside the Wheatstone bridge of section 3.3.

**Worked example 3 (series and parallel).** A $10\,$V source drives a $2\,\mathrm{k\Omega}$ resistor in series with the parallel pair $3\,\mathrm{k\Omega}$ and $6\,\mathrm{k\Omega}$. The pair is $3\cdot6/9 = 2\,\mathrm{k\Omega}$, so the total is $4\,\mathrm{k\Omega}$ and the source current is $2.5\,$mA. The voltage divider puts $10\cdot2/4 = 5\,$V across the pair. Then $5/3 = 1.667\,$mA flows in the $3\,\mathrm{k\Omega}$ branch and $5/6 = 0.833\,$mA in the $6\,\mathrm{k\Omega}$ branch, which add back to $2.5\,$mA as the node law demands.

**Worked example 4 (current divider).** A current source pushes $3\,$mA into a node connected to ground through $1\,\mathrm{k\Omega}$ and $2\,\mathrm{k\Omega}$ in parallel. The pair is $2/3\,\mathrm{k\Omega}$, the node sits at $2\,$V, and the currents are $2\,$mA and $1\,$mA: twice as much through the smaller resistor. The code checks both examples with the general solver.

In [ ]:
# Worked example 3: node 1 is the source's + terminal, node 2 the junction above the parallel pair.
V, I_src = ci.mna(3, resistors=[(1, 2, 2e3), (2, 0, 3e3), (2, 0, 6e3)], voltage_sources=[(1, 0, 10.0)])
print(f"worked example 3: V(junction) = {V[2]:.3f} V,  I(3k) = {V[2] / 3e3 * 1e3:.3f} mA,  I(6k) = {V[2] / 6e3 * 1e3:.3f} mA,  source current = {-I_src[0] * 1e3:.3f} mA")

# Worked example 4: a 3 mA current source from ground into node 1, two resistors from node 1 to ground.
V, _ = ci.mna(2, resistors=[(1, 0, 1e3), (1, 0, 2e3)], current_sources=[(0, 1, 3e-3)])
print(f"worked example 4: V = {V[1]:.3f} V,  I(1k) = {V[1] / 1e3 * 1e3:.3f} mA,  I(2k) = {V[1] / 2e3 * 1e3:.3f} mA")

### 3.3 Nodal analysis

Series and parallel rules run out quickly: a bridge, a ladder with sources at both ends, or a grid of resistors cannot be reduced by them. Nodal analysis works for every linear circuit and is mechanical enough for a computer.

**The recipe.** Pick one node as ground and set its potential to zero (only differences matter, so one potential is free; that is warm-up question 4). The remaining $N - 1$ potentials are the unknowns. At each of these nodes, write the node law with every resistor current expressed by Ohm's law, $(V_a - V_b)/R = g(V_a - V_b)$ with $g = 1/R$ the conductance. Collect the terms: each node gives one linear equation, and together they form
$$\mathbf{G}\mathbf{V} = \mathbf{I},$$
with $\mathbf{I}$ the currents injected into each node by current sources.

**The stamp rule.** The matrix can be written down without forming the equations one by one. A resistor of conductance $g$ between nodes $a$ and $b$ adds $g$ to $G_{aa}$ and to $G_{bb}$, and $-g$ to $G_{ab}$ and to $G_{ba}$. A resistor from $a$ to ground adds $g$ to $G_{aa}$ only, since the ground row and column are dropped. A current source that delivers $I$ into node $a$ adds $I$ to the right-hand side at $a$. So each diagonal entry is the total conductance touching that node, and each off-diagonal entry is minus the conductance joining two nodes. The matrix is symmetric, and for a connected circuit with a ground it is positive definite, so the system has exactly one solution: the uniqueness argument of M2 primer 00b in circuit form.

**Worked example 5 (three nodes by hand).** Nodes 1, 2, 3 and ground. Resistors: $1\,\mathrm{k\Omega}$ from 1 to ground, $1\,\mathrm{k\Omega}$ from 1 to 2, $2\,\mathrm{k\Omega}$ from 2 to ground, $1\,\mathrm{k\Omega}$ from 2 to 3, $1\,\mathrm{k\Omega}$ from 3 to ground. A current source delivers $1\,$mA into node 1. In millisiemens and milliamps, stamping each resistor gives
$$\begin{pmatrix} 2 & -1 & 0 \\ -1 & 2.5 & -1 \\ 0 & -1 & 2 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ V_3 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \\ 0 \end{pmatrix}.$$
Row 2, for instance: node 2 touches $1 + 0.5 + 1 = 2.5\,$mS, and connects to nodes 1 and 3 with $1\,$mS each. Solve by substitution. Row 3 gives $V_3 = V_2/2$. Row 2 gives $-V_1 + 2.5V_2 - V_2/2 = 0$, so $V_1 = 2V_2$. Row 1 gives $4V_2 - V_2 = 1$, so $V_2 = 1/3\,$V, $V_1 = 2/3\,$V, $V_3 = 1/6\,$V. The code builds the same matrix twice, once typed in and once by a stamping loop, and compares both with the library solver.

In [ ]:
# Worked example 5: the matrix typed in (mS), and the same matrix built by stamping.
G_typed = np.array([[2.0, -1, 0], [-1, 2.5, -1], [0, -1, 2]]) * 1e-3
resistors = [(1, 0, 1e3), (1, 2, 1e3), (2, 0, 2e3), (2, 3, 1e3), (3, 0, 1e3)]
G = np.zeros((4, 4))
for a, b, Rk in resistors:
    g = 1.0 / Rk
    G[a, a] += g; G[b, b] += g; G[a, b] -= g; G[b, a] -= g
G = G[1:, 1:]                                          # drop the ground row and column
print("stamped G [mS]:\n", G * 1e3, "\nsame as typed:", np.allclose(G, G_typed))
V_hand = np.linalg.solve(G, np.array([1e-3, 0, 0]))
V_lib, _ = ci.mna(4, resistors=resistors, current_sources=[(0, 1, 1e-3)])
print("by hand:", V_hand, "   emlib.circuits.mna:", V_lib[1:], "   exact: [2/3, 1/3, 1/6]")

**Voltage sources and modified nodal analysis.** A current source fits the recipe directly, because its current goes straight into $\mathbf{I}$. An ideal voltage source does not: it fixes a potential difference, $V_a - V_b = V_s$, and the current through it is whatever the rest of the circuit demands, which Ohm's law cannot supply. Modified nodal analysis (MNA) handles this by adding that current as one more unknown, and the constraint $V_a - V_b = V_s$ as one more equation. The system becomes
$$\begin{pmatrix} \mathbf{G} & \mathbf{B} \\ \mathbf{B}^T & 0 \end{pmatrix}\begin{pmatrix} \mathbf{V} \\ \mathbf{I}_s \end{pmatrix} = \begin{pmatrix} \mathbf{I} \\ \mathbf{V}_s \end{pmatrix},$$
where each column of $\mathbf{B}$ has a $+1$ in row $a$ and a $-1$ in row $b$ of its source. The unknown $I_s$ enters the node law at $a$ as a current leaving the node into the source's $+$ terminal. This is what `emlib.circuits.mna` assembles. A source that is delivering power has current coming out of its $+$ terminal, so the solver reports a negative $I_s$ for it.

**Worked example 6 (a divider driven by a voltage source).** A $10\,$V source from node 1 to ground, $1\,\mathrm{k\Omega}$ from node 1 to node 2, $3\,\mathrm{k\Omega}$ from node 2 to ground. Unknowns $V_1$, $V_2$, $I_s$. With $g_1 = 1\,$mS and $g_2 = 1/3\,$mS:
$$\begin{pmatrix} g_1 & -g_1 & 1 \\ -g_1 & g_1 + g_2 & 0 \\ 1 & 0 & 0 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ I_s \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 10 \end{pmatrix}.$$
The last row says $V_1 = 10$. Row 2 gives $V_2 = g_1V_1/(g_1 + g_2) = 7.5\,$V, the voltage divider. Row 1 gives $I_s = g_1(V_2 - V_1) = -2.5\,$mA: the source delivers $2.5\,$mA.

In [ ]:
g1, g2 = 1 / 1e3, 1 / 3e3
A = np.array([[g1, -g1, 1.0], [-g1, g1 + g2, 0.0], [1.0, 0.0, 0.0]])
x = np.linalg.solve(A, np.array([0.0, 0.0, 10.0]))
print("MNA matrix:\n", A)
print(f"V1 = {x[0]:.3f} V,  V2 = {x[1]:.3f} V,  I_s = {x[2] * 1e3:.3f} mA")
V, I_s = ci.mna(3, resistors=[(1, 2, 1e3), (2, 0, 3e3)], voltage_sources=[(1, 0, 10.0)])
print(f"emlib.circuits.mna: V2 = {V[2]:.3f} V,  I_s = {I_s[0] * 1e3:.3f} mA")

**The Wheatstone bridge.** Four resistors form two voltage dividers fed by the same source: $R_1$ over $R_2$ on the left (junction node 2) and $R_3$ over $R_4$ on the right (junction node 3). A fifth element, historically a galvanometer of resistance $R_g$, joins the two junctions. Before it is connected, the dividers give
$$V_2 = V_s\frac{R_2}{R_1 + R_2}, \qquad V_3 = V_s\frac{R_4}{R_3 + R_4}.$$
If these are equal, connecting the middle branch changes nothing: there is no potential difference across it and no current flows. Setting them equal and cross-multiplying gives the balance condition
$$\frac{R_1}{R_2} = \frac{R_3}{R_4}.$$
That is how the bridge measures an unknown resistance: put it at $R_4$, adjust a calibrated $R_3$ until the galvanometer reads zero, and read off $R_4 = R_3R_2/R_1$. A null reading needs no calibrated meter, and it does not depend on the source voltage.

**Thevenin's theorem.** Any linear network of sources and resistors, seen from two of its terminals, behaves exactly like one ideal source $V_{\text{th}}$ in series with one resistor $R_{\text{th}}$. Two measurements find them. The open-circuit voltage between the terminals is $V_{\text{th}}$, since no current flows through $R_{\text{th}}$. The short-circuit current is $I_{\text{sc}} = V_{\text{th}}/R_{\text{th}}$, so $R_{\text{th}} = V_{\text{th}}/I_{\text{sc}}$. Equivalently, $R_{\text{th}}$ is the resistance between the terminals with every source turned off: voltage sources replaced by wires, current sources by gaps. The theorem holds because the network is linear: the terminal voltage is a linear function of the terminal current, which is a straight line with intercept $V_{\text{th}}$ and slope $-R_{\text{th}}$.

**Worked example 7 (an unbalanced bridge).** $V_s = 10\,$V, $R_1 = 100$, $R_2 = 200$, $R_3 = 300$, $R_4 = 400\,\Omega$, $R_g = 100\,\Omega$. The ratios $1/2$ and $3/4$ differ, so current flows in the middle branch. Seen from the two junctions, the open-circuit voltage is the difference of the dividers:
$$V_{\text{th}} = 10\left(\frac{200}{300} - \frac{400}{700}\right) = 0.9524\,\mathrm{V}.$$
With the source replaced by a wire, the top and bottom nodes are the same node, and each divider becomes two resistors in parallel between that node and its junction: $R_{\text{th}} = 100\parallel200 + 300\parallel400 = 66.67 + 171.43 = 238.10\,\Omega$. So $I_{\text{sc}} = 0.9524/238.10 = 4.000\,$mA and the galvanometer current is $I_g = V_{\text{th}}/(R_{\text{th}} + R_g) = 0.9524/338.10 = 2.817\,$mA. The code computes all three with the solver: the open-circuit voltage with the middle branch removed, the short-circuit current with a $0\,$V source in its place, and the full bridge. It also sweeps $R_4$ to show $I_g$ crossing zero at the balance point $R_4 = 600\,\Omega$ for $R_3 = 300$.

In [ ]:
Vs, R1, R2, R3, R4, Rg = 10.0, 100.0, 200.0, 300.0, 400.0, 100.0
arms = [(1, 2, R1), (2, 0, R2), (1, 3, R3), (3, 0, R4)]
V_open, _ = ci.mna(4, resistors=arms, voltage_sources=[(1, 0, Vs)])
V_th = V_open[2] - V_open[3]
_, I_srcs = ci.mna(4, resistors=arms, voltage_sources=[(1, 0, Vs), (2, 3, 0.0)])   # a 0 V source is a short from 2 to 3
I_sc = I_srcs[1]
V_full, _ = ci.mna(4, resistors=arms + [(2, 3, Rg)], voltage_sources=[(1, 0, Vs)])
I_g = (V_full[2] - V_full[3]) / Rg
print(f"V_th = {V_th:.4f} V   I_sc = {I_sc * 1e3:.3f} mA   R_th = V_th / I_sc = {V_th / I_sc:.2f} ohm   (hand: 238.10)")
print(f"galvanometer current: full solve {I_g * 1e3:.4f} mA,  Thevenin {V_th / (V_th / I_sc + Rg) * 1e3:.4f} mA")

R4s = np.linspace(300, 900, 121)
Ig_sweep = []
for R4k in R4s:
    Vk, _ = ci.mna(4, resistors=[(1, 2, R1), (2, 0, R2), (1, 3, R3), (3, 0, R4k), (2, 3, Rg)], voltage_sources=[(1, 0, Vs)])
    Ig_sweep.append((Vk[2] - Vk[3]) / Rg)
Ig_sweep = np.array(Ig_sweep)
print(f"|I_g| is smallest at R4 = {R4s[np.argmin(np.abs(Ig_sweep))]:.0f} ohm (|I_g| = {np.abs(Ig_sweep).min():.1e} A); balance predicts R3 R2 / R1 = {R3 * R2 / R1:.0f} ohm")
plt.plot(R4s, Ig_sweep * 1e3); plt.axhline(0, color="grey", lw=0.5); plt.axvline(600, color="grey", ls=":")
plt.xlabel("R4 [ohm]"); plt.ylabel("I_g [mA]"); plt.title("the bridge balances at R1/R2 = R3/R4"); plt.show()

### 3.4 Circuits as models

So far the circuit was given. Section 10.3 of the book turns the idea around: a continuous conductor can be modelled by a circuit. Divide a conducting sheet of thickness $t$ and conductivity $\sigma$ into square cells of side $h$ and put a node at the centre of each. Between two neighbouring nodes the current crosses a face of width $h$ and thickness $t$, over a distance $h$, so primer 00a's $R = L/(\sigma A)$ gives a resistor
$$R_{\text{face}} = \frac{h}{\sigma\,h\,t} = \frac{1}{\sigma t}, \qquad g_{\text{face}} = \sigma t,$$
independent of $h$, which is a special property of two dimensions. If $\sigma$ differs between the two cells, the face is two half-cells in series, and the face conductance is the harmonic mean $2\sigma_1\sigma_2/(\sigma_1 + \sigma_2)$ times $t$: the same rule the dielectric solver of module 3 uses for $\epsilon$, and for the same reason. The node law at each node is then the discrete form of $\nabla\cdot(\sigma\nabla V) = 0$. The sheet has become a resistor network, and nodal analysis solves it.

**Worked example 8 (a uniform strip).** A sheet with $\sigma t = 1\,$S, $41$ nodes long and $11$ nodes wide, electrodes on the two short ends at $1\,$V and $0\,$V, insulating long edges. Every row is $40$ unit resistors in series ($40\,\Omega$), and the $11$ rows are in parallel, so the total is $40/11 = 3.636\,\Omega$ and the current $0.275\,$A. That is the continuous result $R = L/(\sigma t W)$ with $L = 40h$ and $W = 11h$, where each row of nodes stands for a strip of width $h$.

The code builds the grid network explicitly, edge by edge, solves it with the general network solver `emlib.circuits.solve_network`, and compares with `emlib.circuits.solve_sheet`, which does the same assembly internally and is what notebook 01 uses. It then cuts a hole in the sheet ($\sigma = 0$ marks missing material, and a face touching it gets zero conductance) and shows the current bending around it.

In [ ]:
def sheet_network(sigma, t=1.0):
    '''Build the resistor network of a sheet explicitly: node numbers, edges and face conductances.'''
    nx, ny = sigma.shape
    h_edges, v_edges = ci.grid_edges(nx, ny)
    def face(s1, s2):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where((s1 > 0) & (s2 > 0), 2 * s1 * s2 / (s1 + s2), 0.0) * t
    edges = np.concatenate([h_edges.reshape(-1, 2), v_edges.reshape(-1, 2)])
    G = np.concatenate([face(sigma[:-1], sigma[1:]).ravel(), face(sigma[:, :-1], sigma[:, 1:]).ravel()])
    fixed = np.zeros((nx, ny), bool); fixed[0], fixed[-1] = True, True
    V_fixed = np.zeros((nx, ny)); V_fixed[0] = 1.0
    V, I_edge = ci.solve_network(nx * ny, edges, G, fixed.ravel(), V_fixed.ravel())
    I_total = I_edge[: (nx - 1) * ny].reshape(nx - 1, ny)[0].sum()
    return V.reshape(nx, ny), I_total

nx, ny = 41, 11
V_net, I_net = sheet_network(np.ones((nx, ny)))
V_sh, _, _, I_sh = ci.solve_sheet(np.ones((nx, ny)))
print(f"uniform strip: network I = {I_net:.4f} A, solve_sheet I = {I_sh:.4f} A, rows in parallel 11/40 = {11 / 40:.4f} A")

sigma = np.ones((nx, ny)); sigma[15:26, 3:8] = 0.0
V_net, I_net = sheet_network(sigma)
V_sh, Ix, Iy, I_sh = ci.solve_sheet(sigma)
print(f"with a hole:   network I = {I_net:.4f} A, solve_sheet I = {I_sh:.4f} A, max |V difference| = {np.abs(V_net - V_sh).max():.1e}")

x, y = np.arange(nx), np.arange(ny)
Jx = np.zeros((nx, ny)); Jx[:-1] += Ix / 2; Jx[1:] += Ix / 2
Jy = np.zeros((nx, ny)); Jy[:, :-1] += Iy / 2; Jy[:, 1:] += Iy / 2
fig, ax = plt.subplots(figsize=(9, 3))
cf = ax.contourf(x, y, V_sh.T, levels=21, cmap="viridis"); fig.colorbar(cf, label="V")
ax.streamplot(x, y, Jx.T, Jy.T, color="w", density=1.2, linewidth=0.7)
ax.add_patch(plt.Rectangle((14.5, 2.5), 11, 5, color="k"))
ax.set_aspect("equal"); ax.set_title("a sheet as a resistor network: current bends around a hole"); plt.show()

**Worked example 9 (the infinite ladder).** A ladder network is a chain of sections, each a series resistor $R$ followed by a shunt resistor $R$ to ground. What is the input resistance of an infinitely long ladder? Since the ladder is infinite, removing the first section leaves the same ladder, with the same input resistance $R_\infty$. So the first section sees $R_\infty$ in parallel with its shunt resistor:
$$R_\infty = R + \frac{R\,R_\infty}{R + R_\infty} \quad\Longrightarrow\quad R_\infty^2 - RR_\infty - R^2 = 0 \quad\Longrightarrow\quad R_\infty = R\,\frac{1 + \sqrt5}{2} = 1.618\,R.$$
A finite ladder of $n$ sections follows the same recursion, $R_k = R + R\,R_{k-1}/(R + R_{k-1})$, starting from $R_1 = 2R$ (one series and one shunt resistor). The values are $2$, $5/3$, $13/8$, $34/21$, ... times $R$: ratios of Fibonacci numbers, converging to the golden ratio. Each additional section shrinks the remaining error by a factor of about $6.9$ (it is $\varphi^4$, with $\varphi$ the golden ratio), so eight sections already agree with the infinite ladder to six significant digits. The ladder is also a model: a long cable with resistance along its length and leakage to ground through its insulation is exactly this network, and the exponential decay of the signal along it is the same convergence seen from the other side.

In [ ]:
R_ladder = [2.0]
for k in range(2, 13):
    R_ladder.append(1.0 + R_ladder[-1] / (1.0 + R_ladder[-1]))
golden = (1 + np.sqrt(5)) / 2
fib = [1, 1]
while len(fib) < 30: fib.append(fib[-1] + fib[-2])
for n, Rn in enumerate(R_ladder, start=1):
    print(f"n = {n:2d}:  R_n / R = {Rn:.8f}   F_(2n+1)/F_(2n) = {fib[2 * n] / fib[2 * n - 1]:.8f}   |R_n - R_inf| / R = {abs(Rn - golden):.1e}")

**When is a lumped model valid?** The circuit model assumes three things. Wires are equipotentials, so their resistance is negligible next to the components (or is drawn as a resistor of its own). Each component is fully described by a relation between its current and the voltage across it, so no charge is stored in the wires and no field leaks from one component to another. And signals travel through the circuit instantly compared with how fast they change, which holds when the circuit is small compared with the wavelength $c/f$: $6000\,$km at $50\,$Hz, $30\,$cm at $1\,$GHz. A mains circuit on a bench satisfies all three easily; a circuit board at gigahertz frequencies does not. Primer 00c, in its section on the assumptions behind circuit models (book section 16.4), returns to these limits once inductance and alternating currents are on the table.

### 3.5 RC circuits

**The capacitor in a circuit.** A capacitor holds charge $+Q$ on one plate and $-Q$ on the other, with potential difference $V = Q/C$ (M2 primer 00c). The current flowing into the $+$ plate is the rate at which $Q$ grows, and an equal current leaves the $-$ plate, so as far as the rest of the circuit can tell, a current flows "through" the capacitor even though no charge crosses the gap:
$$I = \frac{dQ}{dt} = C\,\frac{dV}{dt}.$$
A capacitor cannot change its voltage instantly, because that would need an infinite current. At the moment a switch closes, an uncharged capacitor behaves like a wire (zero voltage); long afterwards, when its voltage has stopped changing, it carries no current and behaves like a gap. Those two limits give the start and end of every RC transient without solving anything.

**Charging.** A battery $V_0$, a resistor $R$ and an uncharged capacitor $C$ in series; the switch closes at $t = 0$. The loop law reads $V_0 - IR - V_C = 0$, and with $I = C\,dV_C/dt$:
$$RC\,\frac{dV_C}{dt} = V_0 - V_C, \qquad V_C(0) = 0.$$
The right-hand side says the capacitor voltage moves towards $V_0$ at a rate proportional to the remaining gap, which is the definition of exponential approach. Separate variables: $dV_C/(V_0 - V_C) = dt/RC$, integrate, and use $V_C(0) = 0$:
$$V_C(t) = V_0\left(1 - e^{-t/RC}\right), \qquad I(t) = \frac{V_0}{R}\,e^{-t/RC}.$$
The product $\tau = RC$ has units of seconds (ohm times farad is volt per ampere times coulomb per volt, which is coulomb per ampere) and is called the time constant. After one $\tau$ the gap has shrunk by a factor $e$, so the capacitor has reached $1 - e^{-1} = 63.2$ percent of $V_0$; after $5\tau$ it is within $e^{-5} = 0.7$ percent. The current starts at $V_0/R$, as if the capacitor were a wire, and decays with the same $\tau$.

**Discharging.** Remove the battery and connect the charged capacitor across $R$. The loop law gives $RC\,dV_C/dt = -V_C$ and $V_C(t) = V_0\,e^{-t/RC}$, with the current flowing out of the $+$ plate. The same time constant governs both directions.

**Worked example 10.** $R = 10\,\mathrm{k\Omega}$, $C = 100\,\mu$F, $V_0 = 5\,$V. The time constant is $10^4\cdot10^{-4} = 1\,$s. The initial current is $5/10^4 = 0.5\,$mA. At $t = 1\,$s, $V_C = 5(1 - e^{-1}) = 3.161\,$V; at $t = 5\,$s, $4.966\,$V. Two capacitors in parallel add their capacitance and double $\tau$; two $100\,\mu$F capacitors in series give $50\,\mu$F and $\tau = 0.5\,$s, by the series and parallel rules of M2 primer 00c.

**Animation.** A battery, a resistor and a capacitor in one loop. When the switch closes, the current arrows start large and fade while charge builds on the plates. Beside the circuit, $V_C(t)$ rises as $1 - e^{-t/RC}$ and $I(t)$ falls as $e^{-t/RC}$, with markers at $t = RC$, where the capacitor has reached 63 percent of the battery voltage, and at $5RC$, where it is essentially full.

In [ ]:
from emlib import show
show.animation("m5_circuits", "CapacitorCharging")

In [ ]:
R, C, V0 = 1e4, 1e-4, 5.0
tau = R * C
t = np.linspace(0.0, 6 * tau, 601)
Vc_num = rk4_ode(lambda tk, v: (V0 - v) / (R * C), 0.0, t)            # charging: RC dV/dt = V0 - V
Vd_num = rk4_ode(lambda tk, v: -v / (R * C), V0, t)                   # discharging: RC dV/dt = -V
print(f"tau = {tau:.2f} s, I(0) = {V0 / R * 1e3:.2f} mA")
print(f"V_C(tau) = {np.interp(tau, t, Vc_num):.3f} V ({np.interp(tau, t, Vc_num) / V0 * 100:.1f} percent),  V_C(5 tau) = {np.interp(5 * tau, t, Vc_num):.3f} V")
print(f"max |rk4 - closed form|: charging {np.abs(Vc_num - ci.rc_charging(t, R, C, V0)).max():.1e} V, discharging {np.abs(Vd_num - V0 * np.exp(-t / tau)).max():.1e} V")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t, Vc_num, label="V_C charging"); axes[0].plot(t, Vd_num, label="V_C discharging")
axes[0].axvline(tau, color="grey", ls=":"); axes[0].axhline(0.632 * V0, color="grey", ls=":")
axes[0].set_xlabel("t [s]"); axes[0].set_ylabel("V [V]"); axes[0].legend(); axes[0].set_title("one time constant: 63 percent of the way")
axes[1].plot(t, (V0 - Vc_num) / R * 1e3); axes[1].set_xlabel("t [s]"); axes[1].set_ylabel("I [mA]"); axes[1].set_title("charging current decays with the same tau")
plt.tight_layout(); plt.show()

**Energy bookkeeping.** While the capacitor charges, a total charge $Q = CV_0$ passes through the battery, which does work $QV_0 = CV_0^2$ on it. The capacitor ends up storing $\tfrac12CV_0^2$ (M2 primer 00c). The rest is dissipated in the resistor:
$$\int_0^\infty I^2R\,dt = \frac{V_0^2}{R}\int_0^\infty e^{-2t/RC}\,dt = \frac{V_0^2}{R}\cdot\frac{RC}{2} = \tfrac12CV_0^2.$$
Exactly half the energy from the battery is lost, whatever the value of $R$. A smaller resistor gives a larger current for a shorter time, and the two effects cancel in the integral. That is warm-up question 6, and it is often a surprise. The loss can only be reduced by charging in many small steps (or with an inductor, primer 00c), so that the voltage across the resistor is never large.

**Worked example 11.** For worked example 10, the battery supplies $CV_0^2 = 10^{-4}\cdot25 = 2.5\,$mJ, the capacitor stores $1.25\,$mJ and the resistor dissipates $1.25\,$mJ. With $R = 1\,\mathrm{k\Omega}$ or $100\,\mathrm{k\Omega}$ the split is the same; only the duration changes.

**Worked example 12 (a camera flash).** A flash capacitor of $330\,\mu$F charged to $300\,$V stores $\tfrac12\cdot330\times10^{-6}\cdot300^2 = 14.85\,$J. Discharged through a flash tube of about $1\,\Omega$, the time constant is $0.33\,$ms and the peak current about $300\,$A: the energy that took seconds to accumulate from a small battery is released in a fraction of a millisecond. Charging it from the battery through the charging circuit cost at least another $14.85\,$J by the half-lost rule, unless the circuit charges in small steps.

The code integrates $I^2R$ numerically for three resistances and compares with $\tfrac12CV_0^2$.

In [ ]:
C, V0 = 1e-4, 5.0
print(f"battery supplies C V0^2 = {C * V0**2 * 1e3:.3f} mJ,  capacitor stores C V0^2 / 2 = {0.5 * C * V0**2 * 1e3:.3f} mJ")
for R in (1e3, 1e4, 1e5):
    tk = np.linspace(0.0, 30 * R * C, 30001)
    I = V0 / R * np.exp(-tk / (R * C))
    print(f"R = {R:8.0f} ohm (tau = {R * C:6.2f} s): dissipated in R = {np.trapezoid(I**2 * R, tk) * 1e3:.4f} mJ")
print(f"camera flash: stored {0.5 * 330e-6 * 300**2:.2f} J, discharge tau through 1 ohm = {330e-6 * 1.0 * 1e3:.2f} ms, peak current {300 / 1.0:.0f} A")

**Worked example 13 (two capacitors and the missing energy).** A capacitor $C_1 = 100\,\mu$F charged to $V_0 = 10\,$V is connected through a resistor $R$ to an uncharged $C_2 = 50\,\mu$F. Charge flows until both have the same voltage. Charge is conserved, $C_1V_0 = (C_1 + C_2)V_f$, so $V_f = V_0C_1/(C_1 + C_2) = 6.667\,$V. The energy before is $\tfrac12C_1V_0^2 = 5.000\,$mJ and after is $\tfrac12(C_1 + C_2)V_f^2 = 3.333\,$mJ. One third has gone, the fraction $C_2/(C_1 + C_2)$, and again it does not depend on $R$: it is dissipated in the resistor whatever its value. With an "ideal" wire of zero resistance the paradox is that the energy still disappears; the resolution is that a real connection always has resistance, and at very small resistance the remaining loss goes into radiation from the current pulse, which the lumped model does not describe. The code integrates the two-capacitor circuit,
$$C_1\frac{dV_1}{dt} = -\frac{V_1 - V_2}{R}, \qquad C_2\frac{dV_2}{dt} = +\frac{V_1 - V_2}{R},$$
and checks the final voltage and the dissipated energy.

In [ ]:
C1, C2, V0, R = 100e-6, 50e-6, 10.0, 1e3
tau2 = R * C1 * C2 / (C1 + C2)                       # the pair relaxes with the series capacitance
t = np.linspace(0.0, 15 * tau2, 3001)
f = lambda tk, y: np.array([-(y[0] - y[1]) / (R * C1), (y[0] - y[1]) / (R * C2)])
y = rk4_ode(f, np.array([V0, 0.0]), t)
dissipated = np.trapezoid((y[:, 0] - y[:, 1]) ** 2 / R, t)
E0, Ef = 0.5 * C1 * V0**2, 0.5 * (C1 + C2) * (V0 * C1 / (C1 + C2)) ** 2
print(f"final voltages {y[-1, 0]:.4f} V and {y[-1, 1]:.4f} V,  hand: {V0 * C1 / (C1 + C2):.4f} V")
print(f"energy before {E0 * 1e3:.3f} mJ, after {Ef * 1e3:.3f} mJ, lost {(E0 - Ef) * 1e3:.3f} mJ,  integral of (V1 - V2)^2 / R = {dissipated * 1e3:.3f} mJ")
plt.plot(t * 1e3, y[:, 0], label="V1"); plt.plot(t * 1e3, y[:, 1], label="V2")
plt.xlabel("t [ms]"); plt.ylabel("V"); plt.legend(); plt.title("charge sharing: both settle at 6.67 V"); plt.show()

### Exercise: the ladder by nodal analysis

Replace the body of `ladder_resistance` so that it builds a ladder of `n` sections (series resistor $R$ from node $k$ to node $k + 1$, then shunt resistor $R$ from node $k + 1$ to ground, for $k = 1, \dots, n$), drives it with a known current source into node 1, and returns the input resistance from the resulting potential of node 1. The check compares with the recursion of worked example 9 for $n = 1$ to $12$ and with the golden ratio for a long ladder.

In [ ]:
def ladder_resistance(n, R):
    '''Input resistance of a ladder of n sections (series R, shunt R), by modified nodal analysis.'''
    resistors = []                                                     # replace from here with your own version
    for k in range(1, n + 1):
        resistors.append((k, k + 1, R))
        resistors.append((k + 1, 0, R))
    V, _ = ci.mna(n + 2, resistors=resistors, current_sources=[(0, 1, 1.0)])   # 1 A into node 1
    return V[1] / 1.0


for n in range(1, 13):
    assert np.isclose(ladder_resistance(n, 1.0), R_ladder[n - 1], rtol=1e-12), n
print("n = 1..12 agree with the recursion; n = 40:", ladder_resistance(40, 1.0), " golden ratio:", golden)
assert abs(ladder_resistance(40, 1.0) - golden) < 1e-10
assert np.isclose(ladder_resistance(40, 470.0), 470.0 * golden)

## 4. Verify

Two checks with closed forms. First, the Wheatstone bridge of worked example 7 solved with every element in place must give the galvanometer current $V_{\text{th}}/(R_{\text{th}} + R_g)$ with $V_{\text{th}}$ and $R_{\text{th}}$ from the divider and parallel formulas, and it must vanish when the bridge is balanced. Second, the energy split of an RC charge: the numerically integrated dissipation and the stored energy must each be half of what the battery supplied, independent of $R$, computed this time from the RK4 solution rather than the closed form.

In [ ]:
def galvanometer_current(R1, R2, R3, R4, Rg, Vs):
    V, _ = ci.mna(4, resistors=[(1, 2, R1), (2, 0, R2), (1, 3, R3), (3, 0, R4), (2, 3, Rg)], voltage_sources=[(1, 0, Vs)])
    return (V[2] - V[3]) / Rg

for (R1, R2, R3, R4, Rg, Vs) in ((100, 200, 300, 400, 100, 10), (470, 1000, 220, 680, 50, 5), (100, 200, 300, 600, 100, 10)):
    V_th = Vs * (R2 / (R1 + R2) - R4 / (R3 + R4))
    R_th = R1 * R2 / (R1 + R2) + R3 * R4 / (R3 + R4)
    num, hand = galvanometer_current(R1, R2, R3, R4, Rg, Vs), V_th / (R_th + Rg)
    print(f"bridge {R1, R2, R3, R4}: solver {num * 1e3:+.6f} mA   Thevenin {hand * 1e3:+.6f} mA")
    assert abs(num - hand) < 1e-12 * max(1e-3, abs(hand))

for R in (1e3, 1e4, 1e5):
    C, V0 = 1e-4, 5.0
    t = np.linspace(0.0, 25 * R * C, 5001)
    Vc = rk4_ode(lambda tk, v: (V0 - v) / (R * C), 0.0, t)
    I = (V0 - Vc) / R
    supplied, dissipated, stored = np.trapezoid(V0 * I, t), np.trapezoid(I**2 * R, t), 0.5 * C * Vc[-1] ** 2
    print(f"R = {R:8.0f}: dissipated / supplied = {dissipated / supplied:.6f},  stored / supplied = {stored / supplied:.6f}")
    assert abs(dissipated / supplied - 0.5) < 1e-4 and abs(stored / supplied - 0.5) < 1e-4

## 5. Explore and rebuild

Explore. Replace the shunt resistors of the ladder by capacitors: a chain of $n$ sections, each a series resistor $R$ and a capacitor $C$ to ground, driven at the input by a step from $0$ to $1\,$V. Write the node equations ($C\,dV_k/dt$ equals the net current into node $k$ through its two resistors), integrate them with `rk4_ode`, and measure the time for the last node to reach half the input voltage for $n = 5, 10, 20, 40$. How does that time grow with $n$? Compare with the number of relaxation sweeps needed on an $n$-point grid in M2 primer 00b, and explain why the two scalings agree: the RC chain is a discretised diffusion equation, and so is Jacobi relaxation.

Rebuild. Close this notebook. On paper, derive Kirchhoff's two laws from $\nabla\cdot\mathbf{J} = 0$ and $\nabla\times\mathbf{E} = 0$; derive the series and parallel rules from them; stamp the conductance matrix of worked example 5 and solve it; derive the balance condition of the bridge; derive the charging curve of an RC circuit and the half-lost energy integral. In a blank editor, write a function that stamps resistors and current sources into $\mathbf{G}$ and solves for the node potentials, then extend it with one voltage source as in worked example 6.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| real source | terminal voltage $V = \mathcal{E} - Ir$ | linear model of a battery |
| single loop | $I = \mathcal{E}/(R + r)$ | one source, one load |
| maximum power transfer | $R = r$, $P_{\max} = \mathcal{E}^2/4r$, efficiency 50 percent | fixed source, variable load |
| node law | $\sum I_{\text{out}} = 0$ at every node | steady currents, wires store no charge |
| loop law | $\sum\Delta V = 0$ around every loop | quasi-static fields, $\nabla\times\mathbf{E} = 0$ |
| series | $R = R_1 + R_2$, same current | |
| parallel | $1/R = 1/R_1 + 1/R_2$, same voltage | |
| voltage divider | $V_2 = V_sR_2/(R_1 + R_2)$ | no load on the middle node |
| current divider | $I_1 = I_sR_2/(R_1 + R_2)$ | |
| nodal analysis | $\mathbf{G}\mathbf{V} = \mathbf{I}$; stamp $\pm g$ into rows and columns $a$, $b$ | linear resistors, ground removed |
| modified nodal analysis | extra unknown $I_s$ and equation $V_a - V_b = V_s$ per voltage source | |
| bridge balance | $R_1/R_2 = R_3/R_4$ | |
| Thevenin | $V_{\text{th}} = V_{\text{oc}}$, $R_{\text{th}} = V_{\text{oc}}/I_{\text{sc}}$ | linear network, two terminals |
| sheet as network | face conductance $\sigma t$; harmonic mean across a change of $\sigma$ | 2D sheet of thickness $t$ |
| infinite ladder | $R_\infty = R(1 + \sqrt5)/2$ | series $R$, shunt $R$ |
| capacitor in a circuit | $I = C\,dV/dt$ | |
| RC charging | $V_C = V_0(1 - e^{-t/RC})$, $I = (V_0/R)e^{-t/RC}$ | step from $0$ to $V_0$ at $t = 0$ |
| RC discharging | $V_C = V_0e^{-t/RC}$ | |
| time constant | $\tau = RC$; 63.2 percent after $\tau$, 99.3 percent after $5\tau$ | |
| charging energy | battery supplies $CV_0^2$, half stored, half dissipated | any $R$ |
| charge sharing | $V_f = C_1V_0/(C_1 + C_2)$, fraction $C_2/(C_1 + C_2)$ lost | |
| lumped model | circuit size much smaller than $c/f$ | |

## 7. Warm-up answers

1. $\mathcal{E} - Ir = V$ gives $r = (9 - 8)/0.5 = 2\,\Omega$.
2. The node law, net current out of every node is zero, comes from charge conservation in steady state, $\nabla\cdot\mathbf{J} = 0$. The loop law, potential changes around any closed loop add to zero, comes from the electrostatic field being conservative, $\nabla\times\mathbf{E} = 0$.
3. Series $R_1 + R_2$; parallel $R_1R_2/(R_1 + R_2)$. The parallel combination is always smaller than the smaller resistor, because adding a path can only let more current through at the same voltage.
4. $N - 1$. Only potential differences are physical, so one node is chosen as ground with $V = 0$; equivalently, the $N$ node equations are not independent, since the currents leaving all the nodes together sum to zero automatically.
5. $\tau = RC$, the time in which the remaining gap $V_0 - V_C$ shrinks by a factor $e$. After one $\tau$ the capacitor is at $1 - 1/e = 63.2$ percent of its final voltage.
6. Half. The battery supplies $CV_0^2$, the capacitor stores $\tfrac12CV_0^2$, and the resistor dissipates the other $\tfrac12CV_0^2$, independent of $R$: a smaller $R$ gives a larger current for a shorter time.
7. When $R_1/R_2 = R_3/R_4$. The two dividers then hold their junctions at the same potential, so there is no voltage across the middle branch and no current in it.